# エネルギーベースモデル

エネルギーベースモデルは、各点に「データらしさの低さ」を表すエネルギーを付けるモデルです。データらしい点のエネルギーを低くし、データらしくない点のエネルギーを高くします。確率は `exp(-E(x))` に比例しますが、全空間で割り直す正規化定数 `Z` が必要になります。

学習では、本物データのエネルギーを下げ、モデルが低エネルギーだと見なした偽物のエネルギーを上げます。この偽物を負例と呼びます。負例は SGLD でエネルギー地形の上を動かして作り、replay buffer で過去の負例も再利用します。低エネルギー領域が本物だけを覆っているか、負例生成が弱くて危険な偽物を見逃していないかを評価します。

In [ ]:
import math
import random
import statistics

random.seed(31)


def sample_real(n):
    xs = []
    for _ in range(n):
        if random.random() < 0.55:
            xs.append(random.gauss(-2.0, 0.42))
        else:
            xs.append(random.gauss(1.6, 0.55))
    return xs

real_data = sample_real(1200)

print('real mean:', round(statistics.mean(real_data), 3))
print('real std:', round(statistics.pstdev(real_data), 3))
print('left ratio:', round(sum(x < 0 for x in real_data) / len(real_data), 3))

1 次元の 2 モード分布を相手にする。エネルギー関数は 2 つの井戸を持つ形にし、`m1, m2` が谷の位置、`log_a1, log_a2` が谷の細さ、`k1, k2` が谷の相対的な深さを受け持つ。谷の位置はサンプルが集まる場所、深さは相対的な選ばれやすさ、細さは分布の広がりに対応する。

In [ ]:
def logsumexp(values):
    m = max(values)
    return m + math.log(sum(math.exp(v - m) for v in values))


def unpack(theta):
    m1, m2, log_a1, log_a2, k1, k2 = theta
    a1 = math.exp(max(-4.0, min(3.0, log_a1)))
    a2 = math.exp(max(-4.0, min(3.0, log_a2)))
    return m1, m2, a1, a2, k1, k2


def energy(x, theta):
    m1, m2, a1, a2, k1, k2 = unpack(theta)
    e1 = a1 * (x - m1) ** 2 + k1
    e2 = a2 * (x - m2) ** 2 + k2
    return -logsumexp([-e1, -e2])


def denergy_dx(x, theta, eps=1e-3):
    return (energy(x + eps, theta) - energy(x - eps, theta)) / (2.0 * eps)


def describe_energy(theta, points=(-3, -2, -1, 0, 1, 2, 3)):
    return [(x, round(energy(x, theta), 3)) for x in points]

initial_theta = [-0.7, 0.8, 0.0, 0.0, 0.0, 0.0]
print(describe_energy(initial_theta))

`p(x) = exp(-E(x)) / Z` と書ける。`Z` は地形全体の `exp(-E)` を積分した値であり、高次元では直接計算しにくい。1 次元なら格子で近似できるため、確率化の意味を数値で見られる。

In [ ]:
def grid_partition(theta, lo=-5.0, hi=5.0, steps=2500):
    dx = (hi - lo) / steps
    total = 0.0
    for i in range(steps):
        x = lo + (i + 0.5) * dx
        total += math.exp(-energy(x, theta)) * dx
    return total


def grid_density(theta, x):
    return math.exp(-energy(x, theta)) / grid_partition(theta)

z0 = grid_partition(initial_theta)
print('approx Z:', round(z0, 3))
print('density samples:', [(x, round(grid_density(initial_theta, x), 4)) for x in [-3, -2, 0, 1.5, 3]])

負例は外から与えられない。SGLD は `-dE/dx` 方向に少し進み、そこへノイズを加える。低エネルギー領域へ寄りながら、同じ場所へ固定されすぎないサンプル列を作る。負例の質が悪いと、モデルが本当に押し上げるべき低エネルギーの偽物を見逃す。

In [ ]:
def sgld(theta, starts, steps=35, step_size=0.045, noise=0.08):
    xs = starts[:]
    for _ in range(steps):
        moved = []
        for x in xs:
            grad = denergy_dx(x, theta)
            x = x - step_size * grad + noise * random.gauss(0.0, 1.0)
            moved.append(max(-5.0, min(5.0, x)))
        xs = moved
    return xs

starts = [random.uniform(-4.5, 4.5) for _ in range(900)]
neg0 = sgld(initial_theta, starts)

print('negative mean:', round(statistics.mean(neg0), 3))
print('negative std:', round(statistics.pstdev(neg0), 3))
print('negative left ratio:', round(sum(x < 0 for x in neg0) / len(neg0), 3))

更新目的は `mean(E(real)) - mean(E(negative))` を小さくすることです。これにより本物のエネルギーは下がり、モデル自身が低く見積もった負例は押し上げられます。分類ラベルを当てるのではなく、データが集まる場所に谷を作り、モデルが間違って低くした場所に山を作り直します。

In [ ]:
def objective(theta, real_batch, neg_batch, reg=0.01):
    real_e = sum(energy(x, theta) for x in real_batch) / len(real_batch)
    neg_e = sum(energy(x, theta) for x in neg_batch) / len(neg_batch)
    penalty = reg * sum(v * v for v in theta)
    return real_e - neg_e + penalty


def finite_grad(theta, real_batch, neg_batch, eps=1e-3):
    grads = []
    for i in range(len(theta)):
        plus = theta[:]
        minus = theta[:]
        plus[i] += eps
        minus[i] -= eps
        g = (objective(plus, real_batch, neg_batch) - objective(minus, real_batch, neg_batch)) / (2.0 * eps)
        grads.append(g)
    return grads


def clip_theta(theta):
    theta[0] = max(-4.0, min(4.0, theta[0]))
    theta[1] = max(-4.0, min(4.0, theta[1]))
    theta[2] = max(-3.0, min(2.0, theta[2]))
    theta[3] = max(-3.0, min(2.0, theta[3]))
    theta[4] = max(-4.0, min(4.0, theta[4]))
    theta[5] = max(-4.0, min(4.0, theta[5]))

batch = real_data[:96]
g = finite_grad(initial_theta, batch, neg0[:96])
print('gradient:', [round(v, 3) for v in g])

replay buffer は、過去の負例を残す記憶領域です。毎回ランダムな初期点から負例を作るだけでは、モデルが以前に低エネルギーと判断した危ない点を忘れやすくなります。buffer から再開すると、過去の弱点を継続して点検できます。

In [ ]:
class ReplayBuffer:
    def __init__(self, max_size=3000):
        self.max_size = max_size
        self.items = []

    def add(self, xs):
        self.items.extend(xs)
        if len(self.items) > self.max_size:
            self.items = self.items[-self.max_size:]

    def sample_starts(self, n, fresh_ratio=0.35):
        starts = []
        for _ in range(n):
            if self.items and random.random() > fresh_ratio:
                starts.append(random.choice(self.items))
            else:
                starts.append(random.uniform(-4.5, 4.5))
        return starts


def train(theta, steps=140, batch_size=96):
    theta = theta[:]
    buffer = ReplayBuffer()
    history = []
    for step in range(steps):
        real_batch = random.sample(real_data, batch_size)
        starts = buffer.sample_starts(batch_size)
        neg_batch = sgld(theta, starts, steps=22)
        buffer.add(neg_batch)
        grads = finite_grad(theta, real_batch, neg_batch)
        lr = 0.035 if step < 80 else 0.018
        for i in range(len(theta)):
            theta[i] -= lr * grads[i]
        clip_theta(theta)
        if step % 20 == 0 or step == steps - 1:
            loss = objective(theta, real_batch, neg_batch)
            history.append((step, loss, theta[:]))
    return theta, history, buffer

trained_theta, history, buffer = train(initial_theta)

for step, loss, theta in history:
    print(step, round(loss, 3), [round(v, 2) for v in theta])

学習後の地形を読むと、実データの 2 つの山の近くでエネルギーが低くなる。生成サンプルは正規化定数を直接使わず、SGLD で低エネルギー領域へ移動させて得る。

In [ ]:
model_samples = sgld(trained_theta, [random.uniform(-4.5, 4.5) for _ in range(1800)], steps=70)

print('trained energy:', describe_energy(trained_theta))
print('real mean/std:', round(statistics.mean(real_data), 3), round(statistics.pstdev(real_data), 3))
print('model mean/std:', round(statistics.mean(model_samples), 3), round(statistics.pstdev(model_samples), 3))
print('real left ratio:', round(sum(x < 0 for x in real_data) / len(real_data), 3))
print('model left ratio:', round(sum(x < 0 for x in model_samples) / len(model_samples), 3))

`Z` の近似は格子幅や範囲に依存する。低エネルギー領域の形が似ていても、確率として正しく正規化できているかは別に確かめる必要がある。

In [ ]:
for steps in [200, 800, 3200]:
    z = grid_partition(trained_theta, steps=steps)
    p_left = 0.0
    dx = 10.0 / steps
    for i in range(steps):
        x = -5.0 + (i + 0.5) * dx
        if x < 0:
            p_left += math.exp(-energy(x, trained_theta)) * dx / z
    print('grid', steps, 'Z =', round(z, 3), 'P(x<0) =', round(p_left, 3))

SGLD の歩数が短すぎると、負例が現在のモデル分布を十分に代表しない。押し上げるべき場所を外すため、地形の更新が偏る。

In [ ]:
def one_step_diagnostics(theta, sgld_steps):
    real_batch = random.sample(real_data, 120)
    starts = [random.uniform(-4.5, 4.5) for _ in range(120)]
    neg_batch = sgld(theta, starts, steps=sgld_steps)
    return {
        'steps': sgld_steps,
        'neg_mean': round(statistics.mean(neg_batch), 3),
        'neg_std': round(statistics.pstdev(neg_batch), 3),
        'objective': round(objective(theta, real_batch, neg_batch), 3),
    }

for sgld_steps in [1, 5, 25, 70]:
    print(one_step_diagnostics(trained_theta, sgld_steps))

EBM は密度の値を直接出すモデルではなく、エネルギー地形を学ぶモデルです。確率として使うには `Z` が必要で、学習には負例生成が必要になります。本物を下げ、モデル由来の負例を上げるという対比をつかむと、スコアベースモデルや拡散モデルで出てくる勾配による生成も同じ流れで見られます。